# Physiology group plots

Converted from `plot_physio.py`. Run cells in order. The last cell starts the interactive scorer / plots.

Data live under `experiments/LevelN/Results_CSV` and `Results_xdf` (not in git). Outputs go to `analysis/outputs/`.


In [ ]:
# Run from repo root or from analysis/. This replaces __file__ used by the .py scripts.
from pathlib import Path
import os, sys
cwd = Path.cwd().resolve()
if (cwd / "analysis").is_dir() and (cwd / "experiments").is_dir():
    REPO_ROOT = cwd
elif cwd.name == "analysis":
    REPO_ROOT = cwd.parent
elif cwd.name == "notebooks":
    REPO_ROOT = cwd.parents[1]
else:
    REPO_ROOT = cwd
_ANALYSIS_DIR = REPO_ROOT / "analysis"
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)
%matplotlib inline


Physiological Group Visualization Script
=========================================
Generates 6 visualizations × 2 level-groups from Hume Band HR data.

Run from the GoNoGoTests directory:
    python3 plot_physio_group.py
    python3 plot_physio_group.py --out my_plots/

Two data groups are always produced automatically:
  • L2L3  — Levels 2 & 3 only, ALL subjects (consistent coverage)
  • L123  — Levels 1, 2 & 3, only the SUBSET of subjects that have L1 HR data

Behavioral performance metric (for vis3 scatter):
  Priority 1 — BAPS_s from BAPS_Subject{N}_results.csv  (if available)
  Priority 2 — APS from Analysis_Subject{N}_trialMappedRecovery.txt

Metrics computed (metric 5 — post-session recovery — excluded):
  1. Mean HR per condition
  2. ΔHR vs Set 1 mean (same level): Set k − Set 1 (vis3/vis4, tables). Vis1: absolute bpm in S2/S3
     with a dashed line at the cohort mean of per-person Set 1 HR.
  3. HR trajectory slope (linear regression, bpm/min)
  4. Within-session HR fluctuation (SD of bpm readings, honestly labelled)
  6. Peak HR during session
  7. ΔHR vs Set 1 vs performance score (BAPS or APS)


In [ ]:
import os
import sys
import shutil
import csv
import glob
import math
import re
import statistics
import argparse
import xml.etree.ElementTree as ET
from datetime import datetime
from collections import defaultdict

os.environ['MPLBACKEND'] = 'Agg'
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np

# ── Constants ─────────────────────────────────────────────────────────────────
COND_COLORS = {
    'Baseline':            '#4C9BE8',
    'Active Distraction':  '#E84C4C',
    'Passive Distraction': '#E8A94C',
}
COND_ORDER  = ['Baseline', 'Active Distraction', 'Passive Distraction']
SET_TO_COND = {1: 'Baseline', 2: 'Active Distraction', 3: 'Passive Distraction'}


# ══════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_hr_from_xml(xml_paths, source='Hume', verbose=True):
    """Return list of {'ts': float, 'value': float} for all HR records."""
    seen = set()
    records = []
    for path in xml_paths:
        if not os.path.exists(path):
            if verbose:
                print(f"  [skip] {path} not found")
            continue
        if verbose:
            print(f"  Loading {os.path.basename(path)} …")
        try:
            root = ET.parse(path).getroot()
        except Exception as e:
            print(f"  [warn] could not parse {path}: {e}")
            continue
        for rec in root.findall(f'.//Record[@sourceName="{source}"]'):
            if rec.get('type') != 'HKQuantityTypeIdentifierHeartRate':
                continue
            date_str = rec.get('startDate', '')
            val_str  = rec.get('value', '')
            if not date_str or not val_str:
                continue
            try:
                dt  = datetime.strptime(date_str[:19], '%Y-%m-%d %H:%M:%S')
                ts  = dt.timestamp()
                key = round(ts)
                if key in seen:
                    continue
                seen.add(key)
                records.append({'ts': ts, 'value': float(val_str)})
            except Exception:
                pass
    records.sort(key=lambda r: r['ts'])
    if verbose:
        print(f"  HR records loaded: {len(records)}")
    return records


def _csv_timestamps(path):
    ts = []
    try:
        with open(path, newline='', encoding='utf-8', errors='replace') as fh:
            for row in csv.DictReader(fh):
                s = (row.get('timestamp') or '').strip()[:19]
                try:
                    ts.append(datetime.strptime(s, '%Y-%m-%d %H:%M:%S').timestamp())
                except Exception:
                    pass
    except Exception:
        pass
    return ts


def get_set_windows(subject_num, level_num, base_path='.'):
    """Return {set_num: (start_ts, end_ts)} from behavioral CSVs."""
    for subj_dir in [f'Subject{subject_num}', f'subject{subject_num}']:
        csv_dir = os.path.join(base_path, f'Level{level_num}',
                               'Results_CSV', subj_dir)
        if not os.path.exists(csv_dir):
            continue
        set_windows = {}
        for set_num in [1, 2, 3]:
            files = (glob.glob(os.path.join(csv_dir, f'*_set{set_num}_*.csv')) or
                     glob.glob(os.path.join(csv_dir, f'*set{set_num}*.csv')))
            ts_list = []
            for f in files:
                ts_list.extend(_csv_timestamps(f))
            if ts_list:
                set_windows[set_num] = (min(ts_list), max(ts_list))
        if set_windows:
            return set_windows
    return {}


def load_aps_scores(base_path='.', verbose=True):
    """
    Load APS scores from Analysis_Subject{N}_trialMappedRecovery.txt.
    Returns {subject: {level: {set: ars_score}}}
    """
    pattern = os.path.join(base_path, 'Analysis_Subject*_trialMappedRecovery.txt')
    set_re  = re.compile(r'Set\s+(\d)\s*:\s*ARS\s*=\s*([\d.]+)')
    lvl_re  = re.compile(r'^Level\s+(\d)\s*:', re.MULTILINE)
    aps = {}
    for fpath in sorted(glob.glob(pattern)):
        m = re.search(r'Subject(\d+)', os.path.basename(fpath))
        if not m:
            continue
        subj = int(m.group(1))
        try:
            content  = open(fpath, encoding='utf-8', errors='replace').read()
            start    = content.find('SUMMARY BY LEVEL AND SET')
            if start == -1:
                continue
            cur_lvl  = None
            subj_data = {}
            for line in content[start:].splitlines():
                lm = lvl_re.match(line.strip())
                if lm:
                    cur_lvl = int(lm.group(1))
                    subj_data[cur_lvl] = {}
                sm = set_re.search(line)
                if sm and cur_lvl is not None:
                    subj_data[cur_lvl][int(sm.group(1))] = float(sm.group(2))
            if subj_data:
                aps[subj] = subj_data
        except Exception:
            pass
    if verbose:
        print(f"  APS scores loaded for {len(aps)} subjects")
    return aps


def load_baps_scores(base_path='.', verbose=True):
    """
    Load BAPS_s scores from BAPS_Subject{N}_results.csv files.
    Returns {subject: {level: {set: baps_s}}}
    """
    pattern = os.path.join(base_path, 'BAPS_Subject*_results.csv')
    baps = {}
    for fpath in sorted(glob.glob(pattern)):
        m = re.search(r'Subject(\d+)', os.path.basename(fpath))
        if not m:
            continue
        subj = int(m.group(1))
        try:
            with open(fpath, newline='', encoding='utf-8', errors='replace') as fh:
                for row in csv.DictReader(fh):
                    lvl = int(row.get('Level', 0) or 0)
                    st  = int(row.get('Set',   0) or 0)
                    val = row.get('BAPS_s', '')
                    if not val or lvl == 0 or st == 0:
                        continue
                    try:
                        baps.setdefault(subj, {}).setdefault(lvl, {})[st] = float(val)
                    except Exception:
                        pass
        except Exception:
            pass
    if verbose and baps:
        print(f"  BAPS_s scores loaded for {len(baps)} subjects "
              f"(subjects: {sorted(baps.keys())})")
    elif verbose:
        print("  No BAPS_Subject*_results.csv found — will use APS scores")
    return baps


# ══════════════════════════════════════════════════════════════════════════════
# DATASET BUILDER
# ══════════════════════════════════════════════════════════════════════════════

def _slope(times, values):
    n = len(times)
    if n < 3:
        return None
    sx  = sum(times);  sy  = sum(values)
    sxx = sum(x*x for x in times)
    sxy = sum(x*y for x, y in zip(times, values))
    denom = n * sxx - sx * sx
    if abs(denom) < 1e-9:
        return None
    return (n * sxy - sx * sy) / denom


def build_dataset(hr_records, perf_data, perf_label,
                  subjects, base_path='.', verbose=True):
    """
    Build one record dict per (subject, level, set).
    perf_data: {subject: {level: {set: score}}}
    perf_label: 'BAPS_s' or 'APS'
    """
    rows = []
    for subj in subjects:
        subj_perf = perf_data.get(subj, {})
        for level in [1, 2, 3]:
            windows = get_set_windows(subj, level, base_path)
            if not windows:
                continue

            set_readings = {}
            set_hr_raw   = {}
            for set_num, (t_start, t_end) in windows.items():
                buf_start = t_start - 120
                buf_end   = t_end   + 120
                matching  = [r for r in hr_records
                             if buf_start <= r['ts'] <= buf_end]
                if not matching:
                    continue
                bpms = [r['value'] for r in matching]
                rel  = [(r['ts'] - t_start) / 60 for r in matching]
                set_readings[set_num] = list(zip(rel, bpms))
                set_hr_raw[set_num]   = bpms

            if not set_hr_raw:
                continue

            baseline_hr = (statistics.mean(set_hr_raw[1])
                           if 1 in set_hr_raw else None)

            for set_num, bpms in set_hr_raw.items():
                if set_num == 0:
                    continue
                cond      = SET_TO_COND.get(set_num, f'Set{set_num}')
                n         = len(bpms)
                mean_hr   = statistics.mean(bpms)
                sd_hr     = statistics.stdev(bpms) if n >= 2 else 0.0
                peak_hr   = max(bpms)
                elevation = ((mean_hr - baseline_hr)
                             if baseline_hr is not None else None)
                readings  = set_readings.get(set_num, [])
                times_min = [r[0] for r in readings]
                vals_bpm  = [r[1] for r in readings]
                slp       = _slope(times_min, vals_bpm)
                perf      = subj_perf.get(level, {}).get(set_num)

                rows.append({
                    'subject':    subj,
                    'level':      level,
                    'set':        set_num,
                    'condition':  cond,
                    'hr_mean':    mean_hr,
                    'hr_sd':      sd_hr,
                    'hr_peak':    peak_hr,
                    'n_readings': n,
                    'baseline_hr':  baseline_hr,
                    'hr_elevation': elevation,
                    'slope':      slp,
                    'perf':       perf,     # BAPS_s or APS depending on source
                    'perf_label': perf_label,
                    'readings':   readings,
                })
    return rows


# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _save(fig, path):
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: {path}')


def _aggregate_elevation(rows, cond):
    """Per-subject mean ΔHR (Set k − Set 1) for one condition, averaged across levels."""
    by_subj = defaultdict(list)
    for r in rows:
        if r['condition'] == cond and r['hr_elevation'] is not None:
            by_subj[r['subject']].append(r['hr_elevation'])
    return {s: statistics.mean(v) for s, v in by_subj.items() if v}


def _aggregate_hr_mean(rows, cond):
    """Per-subject mean HR (bpm) for one condition, averaged across levels in rows."""
    by_subj = defaultdict(list)
    for r in rows:
        if r['condition'] == cond and r.get('hr_mean') is not None:
            by_subj[r['subject']].append(r['hr_mean'])
    return {s: statistics.mean(v) for s, v in by_subj.items() if v}


def _jitter(n, width=0.15, seed=42):
    rng = np.random.default_rng(seed)
    return rng.uniform(-width, width, n)


def _group_info(rows):
    """Return (n_subjects, level_set_str) for a row subset."""
    subjs  = {r['subject'] for r in rows}
    levels = sorted({r['level'] for r in rows})
    lvl_str = '+'.join(f'L{l}' for l in levels)
    return len(subjs), lvl_str


# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION 1 — Box plot + individual points
# ══════════════════════════════════════════════════════════════════════════════

def vis1_boxplot(rows, out_dir, tag):
    hr_s1 = _aggregate_hr_mean(rows, 'Baseline')
    hr_s2 = _aggregate_hr_mean(rows, 'Active Distraction')
    hr_s3 = _aggregate_hr_mean(rows, 'Passive Distraction')
    common = sorted(set(hr_s1) & set(hr_s2) & set(hr_s3))
    if not common:
        print('  [warn] vis1_boxplot: no subjects with Set 1 + Set 2 + Set 3 HR; skipping.')
        return

    vals1 = [hr_s1[s] for s in common]
    vals2 = [hr_s2[s] for s in common]
    vals3 = [hr_s3[s] for s in common]
    mean_s1 = statistics.mean(vals1)
    mean_s2 = statistics.mean(vals2)
    mean_s3 = statistics.mean(vals3)

    conds = ['Baseline', 'Active Distraction', 'Passive Distraction']
    display_labels = ['Set 1', 'Set 2', 'Set 3']
    data_by_cond = [vals1, vals2, vals3]

    # Boxes with a clear gap between them; cohort mean text sits in each gap (S1, S2)
    # and tight to the right of the last box (S3) to avoid trailing whitespace.
    box_w = 0.28
    gap = 0.17
    step = box_w + gap
    x0 = 0.42
    x_centers = [x0 + i * step for i in range(3)]
    half = box_w / 2
    all_points = list(zip(x_centers, [vals1, vals2, vals3],
                          [common] * 3, conds))

    fig, ax = plt.subplots(figsize=(8.0, 6.0))
    bp = ax.boxplot(data_by_cond, positions=x_centers, widths=box_w,
                    patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2.5))
    for patch, cond in zip(bp['boxes'], conds):
        patch.set_facecolor(COND_COLORS[cond])
        patch.set_alpha(0.45)

    jit_w = min(0.11, 0.35 * box_w)
    for x_pos, vals, subjs, cond in all_points:
        jit = _jitter(len(vals), width=jit_w)
        ax.scatter(x_pos + jit, vals, color=COND_COLORS[cond],
                   edgecolors='white', linewidths=0.6, s=55, zorder=4, alpha=0.85)
        if vals and np.nanstd(vals) > 0:
            q1, q3 = np.percentile(vals, [25, 75])
            iqr = q3 - q1
            for idx, (jv, v, s) in enumerate(zip(jit, vals, subjs)):
                if v > q3 + 1.5 * iqr or v < q1 - 1.5 * iqr:
                    side = 7 if idx % 2 == 0 else -18
                    ax.annotate(f'S{s}', (x_pos + jv, v),
                                textcoords='offset points', xytext=(side, 2),
                                fontsize=8, fontweight='bold', color='#222222')

    means = [mean_s1, mean_s2, mean_s3]
    ax.scatter(
        x_centers, means,
        marker='D', s=95, facecolors='black', edgecolors='none',
        linewidths=0, zorder=6,
    )
    # Mean values: centered in the gap between boxes (Set 1 & 2); Set 3 tight right of last box.
    for i, m in enumerate(means):
        if i < 2:
            x_txt = (x_centers[i] + x_centers[i + 1]) / 2.0
            ha = 'center'
        else:
            x_txt = x_centers[2] + half + 0.02
            ha = 'left'
        ax.annotate(
            f'{m:.1f}',
            xy=(x_txt, m),
            ha=ha,
            va='center',
            fontsize=9,
            fontweight='bold',
            color='black',
            zorder=7,
            clip_on=False,
        )

    legend_handle = Line2D(
        [0], [0],
        linestyle='None',
        marker='D',
        markersize=9,
        markerfacecolor='black',
        markeredgecolor='none',
        markeredgewidth=0,
        label='Cohort mean across sets',
    )
    ax.legend(handles=[legend_handle], loc='best', fontsize=11, framealpha=0.92)

    ax.set_xticks(x_centers)
    ax.set_xticklabels(display_labels, fontsize=12, fontweight='bold')
    ax.set_xlim(
        x_centers[0] - half - 0.025,
        x_centers[-1] + half + 0.18,
    )
    ax.set_ylabel('Mean HR during block (bpm)', fontsize=12, fontweight='bold')
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(axis='y', alpha=0.3)
    out_path = os.path.join(out_dir, f'physio_vis1_boxplot_{tag}.png')
    _save(fig, out_path)
    # Backward-compatible filename for slides that still reference this name.
    if tag == 'L2L3_all':
        alt = os.path.join(out_dir, 'physio_vis1_boxplot_elevation.png')
        try:
            shutil.copy2(out_path, alt)
            print(f'  Saved: {alt}')
        except OSError as e:
            print(f'  [warn] could not copy to {alt}: {e}')


# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION 2 — Individual trajectory lines
# ══════════════════════════════════════════════════════════════════════════════

def vis2_trajectory(rows, out_dir, tag):
    n_subj, lvl_str = _group_info(rows)
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), sharey=True)
    fig.suptitle(f'HR Trajectories Within Session  ·  {lvl_str}  ·  N = {n_subj} participants\n'
                 'Thin lines = individual participants  ·  Thick line = group mean',
                 fontsize=11, fontweight='bold', y=1.02)

    for ax, cond in zip(axes, COND_ORDER):
        color     = COND_COLORS[cond]
        cond_rows = [r for r in rows
                     if r['condition'] == cond and len(r['readings']) >= 2]
        all_bins  = defaultdict(list)

        for r in cond_rows:
            times = [t for t, _ in r['readings']]
            bpms  = [b for _, b in r['readings']]
            ax.plot(times, bpms, color=color, alpha=0.2, linewidth=0.9)
            for t, b in zip(times, bpms):
                all_bins[round(t / 5) * 5].append(b)

        if all_bins:
            xs = sorted(all_bins)
            ys = [statistics.mean(all_bins[x]) for x in xs]
            ax.plot(xs, ys, color=color, linewidth=2.5, zorder=5,
                    label='Group mean')

        ax.set_title(cond, fontsize=10, fontweight='bold', color=color)
        ax.set_xlabel('Minutes into session', fontsize=9)
        ax.grid(alpha=0.25)
        ax.set_xlim(left=-2)

    axes[0].set_ylabel('Heart Rate (bpm)', fontsize=10)
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, f'physio_vis2_trajectories_{tag}.png'))


# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION 3 — Scatter ΔHR vs Set 1 vs performance
# ══════════════════════════════════════════════════════════════════════════════

def vis3_scatter_perf(rows, out_dir, tag):
    n_subj, lvl_str = _group_info(rows)
    perf_label = next((r['perf_label'] for r in rows if r.get('perf_label')),
                      'Performance')

    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
    fig.suptitle(f'ΔHR vs Set 1 vs {perf_label}  [{lvl_str}, N={n_subj}]',
                 fontsize=13, fontweight='bold', y=1.01)

    for ax, cond in zip(axes, ['Active Distraction', 'Passive Distraction']):
        plot_rows = [r for r in rows
                     if r['condition'] == cond
                     and r['hr_elevation'] is not None
                     and r['perf'] is not None]

        if not plot_rows:
            ax.text(0.5, 0.5, f'No paired HR + {perf_label} data',
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_title(cond, fontsize=10)
            continue

        xs = np.array([r['hr_elevation'] for r in plot_rows])
        ys = np.array([r['perf']          for r in plot_rows])

        ax.scatter(xs, ys, color=COND_COLORS[cond], s=60,
                   edgecolors='white', linewidths=0.7, alpha=0.85, zorder=4)
        for r, x, y in zip(plot_rows, xs, ys):
            ax.annotate(f"S{r['subject']}", (x, y),
                        textcoords='offset points', xytext=(4, 3),
                        fontsize=5.5, color='#555555')

        r_str = ''
        if len(xs) >= 3:
            m, b = np.polyfit(xs, ys, 1)
            xl = np.array([xs.min(), xs.max()])
            ax.plot(xl, m * xl + b, color='#444444', linewidth=1.5,
                    linestyle='--', alpha=0.7)
            if xs.std() > 0 and ys.std() > 0:
                r_val = np.corrcoef(xs, ys)[0, 1]
                r_str = f'  |  r = {r_val:.2f},  n = {len(xs)}'

        med_x, med_y = np.median(xs), np.median(ys)
        ax.axvline(med_x, color='gray', linestyle=':', linewidth=1, alpha=0.6)
        ax.axhline(med_y, color='gray', linestyle=':', linewidth=1, alpha=0.6)
        ax.axvline(0, color='black', linewidth=0.7, alpha=0.4)

        # Quadrant labels — placed at fixed corners with small translucent boxes
        qkw = dict(transform=ax.transAxes, fontsize=6.5,
                   bbox=dict(boxstyle='round,pad=0.2', alpha=0.55))
        ax.text(0.01, 0.99, 'Resilient\n(High score, Low HR)',
                va='top', color='#1A7A3C', **qkw)
        ax.text(0.99, 0.99, 'Hidden Struggle\n(High score, High HR)',
                va='top', ha='right', color='#C05000', **qkw)
        ax.text(0.01, 0.01, 'Disengaged\n(Low score, Low HR)',
                va='bottom', color='#888888', **qkw)
        ax.text(0.99, 0.01, 'Overwhelmed\n(Low score, High HR)',
                va='bottom', ha='right', color='#C0392B', **qkw)

        ax.set_xlabel('Δ HR vs Set 1 mean (bpm)', fontsize=10)
        ax.set_ylabel(f'{perf_label} Score (%)', fontsize=10)
        ax.set_title(f'{cond}{r_str}', fontsize=11, fontweight='bold',
                     color=COND_COLORS[cond])
        ax.grid(alpha=0.25)

    plt.tight_layout()
    _save(fig, os.path.join(out_dir, f'physio_vis3_scatter_{tag}.png'))


# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION 4 — Heatmap
# ══════════════════════════════════════════════════════════════════════════════

def vis4_heatmap(rows, out_dir, tag):
    n_subj, lvl_str = _group_info(rows)
    subjects = sorted({r['subject'] for r in rows})
    conds    = ['Active Distraction', 'Passive Distraction']
    matrix   = np.full((len(subjects), len(conds)), np.nan)

    for j, cond in enumerate(conds):
        elev = _aggregate_elevation(rows, cond)
        for i, s in enumerate(subjects):
            if s in elev:
                matrix[i, j] = elev[s]

    fig, ax = plt.subplots(figsize=(6, max(6, len(subjects) * 0.35)))
    abs_max = np.nanmax(np.abs(matrix)) if not np.all(np.isnan(matrix)) else 1
    im = ax.imshow(matrix, aspect='auto',
                   cmap='RdBu_r', vmin=-abs_max, vmax=abs_max)

    ax.set_xticks(range(len(conds)))
    ax.set_xticklabels(conds, fontsize=10)
    ax.set_yticks(range(len(subjects)))
    ax.set_yticklabels([f'S{s}' for s in subjects], fontsize=7)
    ax.set_title(f'Δ HR vs Set 1 mean (bpm)\n'
                 f'{lvl_str}  ·  N = {n_subj}  ·  Red = faster than Set 1  ·  Blue = slower than Set 1',
                 fontsize=11, fontweight='bold')

    for i in range(len(subjects)):
        for j in range(len(conds)):
            v = matrix[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:+.0f}', ha='center', va='center',
                        fontsize=6,
                        color='white' if abs(v) > abs_max * 0.6 else 'black')

    plt.colorbar(im, ax=ax, label='Δ HR vs Set 1 (bpm)', shrink=0.6)
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, f'physio_vis4_heatmap_{tag}.png'))


# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION 5 — Paired line plot
# ══════════════════════════════════════════════════════════════════════════════

def vis5_paired_lines(rows, out_dir, tag):
    n_subj, lvl_str = _group_info(rows)
    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), sharey=True)
    fig.suptitle(f'Baseline vs Distraction HR — Paired Comparison\n'
                 f'{lvl_str}  ·  N = {n_subj} participants  ·  '
                 f'Red = HR increased  ·  Blue = HR decreased  ·  '
                 f'Bold black = group mean',
                 fontsize=11, fontweight='bold', y=1.01)

    for ax, cond in zip(axes, ['Active Distraction', 'Passive Distraction']):
        base_by_subj = defaultdict(list)
        dist_by_subj = defaultdict(list)
        for r in rows:
            if r['baseline_hr'] is None:
                continue
            base_by_subj[r['subject']].append(r['baseline_hr'])
            if r['condition'] == cond:
                dist_by_subj[r['subject']].append(r['hr_mean'])

        paired = []
        for s in sorted(base_by_subj):
            if s in dist_by_subj:
                b = statistics.mean(base_by_subj[s])
                d = statistics.mean(dist_by_subj[s])
                paired.append((s, b, d))

        jit = _jitter(len(paired), width=0.08)
        for (s, b, d), j in zip(paired, jit):
            color = '#E84C4C' if d > b else '#4C9BE8'
            ax.plot([0 + j, 1 + j], [b, d], color=color,
                    alpha=0.5, linewidth=1.2)
            ax.scatter([0 + j, 1 + j], [b, d], color=color,
                       s=20, zorder=4, alpha=0.8)

        if paired:
            mean_b = statistics.mean(p[1] for p in paired)
            mean_d = statistics.mean(p[2] for p in paired)
            ax.plot([0, 1], [mean_b, mean_d], color='black',
                    linewidth=2.5, zorder=5)
            ax.scatter([0, 1], [mean_b, mean_d], color='black', s=60, zorder=6)

        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Baseline', cond], fontsize=10)
        ax.set_title(cond, fontsize=11, fontweight='bold',
                     color=COND_COLORS[cond])
        ax.grid(axis='y', alpha=0.25)

    axes[0].set_ylabel('Heart Rate (bpm)', fontsize=10)
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, f'physio_vis5_paired_{tag}.png'))


# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION 6 — Slope comparison bar chart
# ══════════════════════════════════════════════════════════════════════════════

def vis6_slope(rows, out_dir, tag):
    n_subj, lvl_str = _group_info(rows)
    subjects  = sorted({r['subject'] for r in rows if r['slope'] is not None})
    n_subj_s  = len(subjects)
    bar_width = 0.25
    x         = np.arange(n_subj_s)

    fig, ax = plt.subplots(figsize=(max(12, n_subj_s * 0.65), 6.5))
    for ci, cond in enumerate(COND_ORDER):
        slopes = []
        for s in subjects:
            cond_rows = [r for r in rows
                         if r['subject'] == s and r['condition'] == cond
                         and r['slope'] is not None]
            slopes.append(statistics.mean(r['slope'] for r in cond_rows)
                          if cond_rows else np.nan)

        offset = (ci - 1) * bar_width
        colors = ['#E84C4C' if (not math.isnan(v) and v > 0) else
                  '#4C9BE8' if (not math.isnan(v)) else
                  'lightgray'
                  for v in slopes]
        ax.bar(x + offset, slopes, bar_width, color=colors, alpha=0.75,
               edgecolor='white', linewidth=0.5)

    ax.axhline(0, color='black', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels([f'S{s}' for s in subjects], fontsize=7.5, rotation=45,
                       ha='right')
    ax.set_ylabel('HR Slope (bpm / minute)', fontsize=10)
    ax.set_title(f'HR Trajectory Slope Per Participant\n'
                 f'{lvl_str}  ·  N = {n_subj}  ·  '
                 f'Positive (red) = rising  ·  Negative (blue) = falling / habituating',
                 fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.25)
    legend_patches = [mpatches.Patch(color=COND_COLORS[c], alpha=0.75, label=c)
                      for c in COND_ORDER]
    ax.legend(handles=legend_patches, fontsize=8, loc='upper right')
    plt.tight_layout()
    _save(fig, os.path.join(out_dir, f'physio_vis6_slopes_{tag}.png'))


# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════════════════

def print_summary(rows, label):
    print(f'\n{"=" * 70}')
    print(f'GROUP: {label}')
    n_subj, lvl_str = _group_info(rows)
    print(f'Subjects: {n_subj}  |  Levels: {lvl_str}  |  Records: {len(rows)}')
    print(f'{"─" * 70}')
    hdr = (f"{'Condition':<22} {'Mean HR':>8} {'Δ vs S1':>10} "
           f"{'SD (fluct)':>11} {'Peak HR':>8} {'N subj':>7}")
    print(hdr)
    print('─' * 70)
    for cond in COND_ORDER:
        cr = [r for r in rows if r['condition'] == cond]
        if not cr:
            continue
        elevs = [r['hr_elevation'] for r in cr if r['hr_elevation'] is not None]
        elev_str = f'{statistics.mean(elevs):+.1f}' if elevs else '   N/A'
        print(f"  {cond:<20} "
              f"{statistics.mean(r['hr_mean'] for r in cr):>7.1f}  "
              f"{elev_str:>9}  "
              f"{statistics.mean(r['hr_sd'] for r in cr):>10.1f}  "
              f"{statistics.mean(r['hr_peak'] for r in cr):>7.1f}  "
              f"{len({r['subject'] for r in cr}):>6}")
    print('=' * 70)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════


In [ ]:
def main():
    parser = argparse.ArgumentParser(
        description='Physiological group visualization — AttentionSpan study')
    parser.add_argument('--base', default='.',
                        help='GoNoGoTests directory (default: .)')
    parser.add_argument('--out', default=None,
                        help='Output folder (default: physio_plots/)')
    parser.add_argument('--subjects', type=int, nargs='+',
                        help='Subject numbers (default: 2-38)')
    parser.add_argument('--source', default='Hume')
    parser.add_argument('--quiet', action='store_true')
    args    = parser.parse_args()
    verbose = not args.quiet

    base_path = args.base
    out_dir   = args.out or os.path.join(base_path, 'physio_plots')
    os.makedirs(out_dir, exist_ok=True)
    print(f'Output folder: {out_dir}/')

    # ── Discover XML files ────────────────────────────────────────────────────
    xml_paths = sorted(glob.glob(os.path.join(base_path, '*export*.xml')))
    if not xml_paths:
        xml_paths = sorted(glob.glob('*export*.xml'))
    if not xml_paths:
        print('ERROR: No *export*.xml files found. Run from GoNoGoTests '
              'or pass --base.')
        return 1
    print(f'Found {len(xml_paths)} XML file(s): '
          f'{[os.path.basename(p) for p in xml_paths]}')

    # ── Load HR data ──────────────────────────────────────────────────────────
    print('\nStep 1: Loading HR data from XML …')
    hr_records = load_hr_from_xml(xml_paths, source=args.source,
                                  verbose=verbose)
    if not hr_records:
        print('ERROR: No HR records loaded.')
        return 1

    # ── Load performance scores (BAPS preferred, APS fallback) ───────────────
    print('\nStep 2: Loading performance scores …')
    baps_data = load_baps_scores(base_path, verbose=verbose)
    aps_data  = load_aps_scores(base_path,  verbose=verbose)

    # Merge: BAPS where available, APS elsewhere
    perf_data  = {}
    perf_label = 'APS'
    for subj in set(list(baps_data.keys()) + list(aps_data.keys())):
        perf_data[subj] = {}
        for level in [1, 2, 3]:
            perf_data[subj][level] = {}
            baps_lvl = baps_data.get(subj, {}).get(level, {})
            aps_lvl  = aps_data.get(subj,  {}).get(level, {})
            for set_num in [1, 2, 3]:
                if set_num in baps_lvl:
                    perf_data[subj][level][set_num] = baps_lvl[set_num]
                    perf_label = 'BAPS_s'     # at least one BAPS score exists
                elif set_num in aps_lvl:
                    perf_data[subj][level][set_num] = aps_lvl[set_num]

    if baps_data:
        print(f'  Using BAPS_s for {len(baps_data)} subject(s); '
              f'APS for the rest.')
    else:
        print(f'  Using APS scores for all subjects.')

    # ── Build full dataset ────────────────────────────────────────────────────
    subjects = args.subjects or list(range(2, 39))
    print(f'\nStep 3: Building dataset for {len(subjects)} subjects …')
    all_rows = build_dataset(hr_records, perf_data, perf_label,
                             subjects, base_path=base_path, verbose=verbose)

    if not all_rows:
        print('ERROR: No dataset rows built.')
        return 1

    # ── Split into two level groups ───────────────────────────────────────────
    # Group A: Levels 2+3 only — all subjects (consistent coverage)
    rows_l23 = [r for r in all_rows if r['level'] in [2, 3]]

    # Group B: All 3 levels — only subjects that have L1 HR data
    subj_with_l1 = {r['subject'] for r in all_rows if r['level'] == 1}
    rows_l123    = [r for r in all_rows
                    if r['subject'] in subj_with_l1]   # keeps L1+L2+L3

    print(f'\n  Group L2+L3 (all subjects):  '
          f'{len({r["subject"] for r in rows_l23})} subjects, '
          f'{len(rows_l23)} records')
    print(f'  Group L1+L2+L3 (subset with L1 data):  '
          f'{len(subj_with_l1)} subjects '
          f'({sorted(subj_with_l1)}), '
          f'{len(rows_l123)} records')

    print_summary(rows_l23,  'L2+L3 — All subjects')
    print_summary(rows_l123, 'L1+L2+L3 — Subset with Level 1 HR data')

    # ── Generate all 6 visualizations × 2 groups ─────────────────────────────
    groups = [
        (rows_l23,  'L2L3_all',    'Levels 2+3, All Subjects'),
        (rows_l123, 'L123_subset', 'Levels 1+2+3, Subset With L1 Data'),
    ]

    print('\nStep 4: Generating visualizations …')
    total_saved = 0
    for rows, tag, label in groups:
        if not rows:
            print(f'  [skip] No data for group: {label}')
            continue
        print(f'\n  ── {label} ──')
        plots = [
            ('vis1 — Box + jitter',     lambda r=rows, t=tag: vis1_boxplot(r, out_dir, t)),
            ('vis2 — Trajectories',     lambda r=rows, t=tag: vis2_trajectory(r, out_dir, t)),
            ('vis3 — Scatter vs perf',  lambda r=rows, t=tag: vis3_scatter_perf(r, out_dir, t)),
            ('vis4 — Heatmap',          lambda r=rows, t=tag: vis4_heatmap(r, out_dir, t)),
            ('vis5 — Paired lines',     lambda r=rows, t=tag: vis5_paired_lines(r, out_dir, t)),
            ('vis6 — Slopes',           lambda r=rows, t=tag: vis6_slope(r, out_dir, t)),
        ]
        for name, fn in plots:
            try:
                fn()
                total_saved += 1
            except Exception as e:
                import traceback
                print(f'  [warn] {name} ({tag}) failed: {e}')
                traceback.print_exc()

    pngs = sorted(f for f in os.listdir(out_dir) if f.endswith('.png'))
    print(f'\nDone — {len(pngs)} PNG files in {out_dir}/')
    for p in pngs:
        print(f'  {p}')
    return 0


In [ ]:
# Run this cell to execute the script
sys.exit(main())
